# 05 — Melnikov Function and Chaos Threshold

Plot $M(t_0)$ analytically and numerically; compute $F_{\mathrm{crit}}$ vs $\omega$ and vs $\delta$; validate by direct simulation. Note the spec's $F_{\mathrm{crit}}$ formula is corrected here — the leading factor is $c_1^{3/2}$ (not $c_1$).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from steering.params import ModelParams, ForcingParams
from steering.models import (
    DuffingModel,
    BesselSteeringModel,
    ContinuousPFLModel,
    DiscretePFLModel,
    FullCircuitModel,
)
from steering.dynamics import VelocityDynamics, AccelerationDynamics
from steering.integrator import Simulation
from steering.visualization.style import use_paper_style

use_paper_style()


In [ ]:
from steering.analysis.melnikov import (
    melnikov_analytical,
    melnikov_critical_forcing,
    melnikov_critical_forcing_numerical,
    melnikov_numerical,
)
from steering.analysis.homoclinic import numerical_homoclinic

bessel = BesselSteeringModel()
p = ModelParams(kappa_h=2.0, kappa_g=2.0, delta=1.4)
c1, c3 = bessel.taylor_coefficients(p)
print(f'c1={c1:.3f}, c3={c3:.3f}')


## Analytical $M(t_0)$ for several $(F, \omega)$

In [ ]:
gamma = 0.05
t0 = np.linspace(0, 4*np.pi, 401)
fig, ax = plt.subplots(figsize=(7, 4))
for F in [0.05, 0.10, 0.20]:
    for omega in [1.0]:
        M = melnikov_analytical(c1, c3, gamma, omega, F, t0)
        ax.plot(t0, M, label=fr'$F={F}, \omega={omega}$')
ax.axhline(0, color='0.5', lw=0.5)
ax.set_xlabel(r'$t_0$'); ax.set_ylabel(r'$M(t_0)$')
ax.legend(); plt.show()


## $F_{\mathrm{crit}}(\omega)$ — Duffing vs Bessel

In [ ]:
omegas = np.linspace(0.2, 4.0, 30)
F_duff = np.array([melnikov_critical_forcing(c1, c3, gamma, om) for om in omegas])
homo = numerical_homoclinic(bessel, p)
F_bess = np.array([melnikov_critical_forcing_numerical(bessel, p, gamma, om, homoclinic=homo) for om in omegas])
fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(omegas, F_duff, label='Duffing analytical')
ax.semilogy(omegas, F_bess, '--', label='Bessel numerical')
ax.set_xlabel(r'$\omega$'); ax.set_ylabel(r'$F_{\mathrm{crit}}$')
ax.legend(); plt.show()


## $F_{\mathrm{crit}}(\delta)$ at fixed $\omega$

Across $\delta$ values where the system is bistable. The minimum is the most chaos-susceptible separation.

In [ ]:
deltas = np.linspace(1.33, 1.55, 18)
F_duff = []
for d in deltas:
    pp = p.replace(delta=d)
    c1d, c3d = bessel.taylor_coefficients(pp)
    if c1d > 0 and c3d < 0:
        F_duff.append(melnikov_critical_forcing(c1d, c3d, gamma, 1.0))
    else:
        F_duff.append(np.nan)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(deltas, F_duff)
ax.set_xlabel(r'$\delta$'); ax.set_ylabel(r'$F_{\mathrm{crit}}$')
plt.show()


## Direct simulation validation

Pick $\omega=1$, $\gamma=0.05$, and three $F$ levels: below, near, and above $F_{\mathrm{crit}}$. Run long sims and inspect the Poincaré section.

In [ ]:
from steering.analysis.poincare import stroboscopic_section
F_crit = melnikov_critical_forcing(c1, c3, gamma, 1.0)
print(f'F_crit (Duffing) = {F_crit:.4f}')
dyn = AccelerationDynamics(model=bessel, gamma=gamma)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, F in zip(axes, [0.5*F_crit, 1.5*F_crit, 4.0*F_crit]):
    forc = ForcingParams(F=F, omega=1.0)
    sim = Simulation(dyn, p, forc, rtol=1e-9, atol=1e-11)
    T = 2*np.pi
    res = sim.run(np.array([0.5, 0.0]), (0.0, 800*T), dense_output=True)
    sec = stroboscopic_section(res, omega=1.0, transient_periods=200)
    ax.plot(sec[:, 0], sec[:, 1], 'k.', ms=1.0, alpha=0.5)
    ax.set_title(fr'$F={F:.3f}$')
    ax.set_xlabel(r'$\theta$')
axes[0].set_ylabel(r'$v$'); plt.show()
